# Governance Role Assigning Hierarchical Clustering Framework  
## Unsupervised Learning + Reinforcement Learning

**Project theme:** This notebook applies text clustering to cybersecurity salary/job role data from Kaggle to build a **Governance Role Assigning Clustering Framework**. The goal is to identify natural groupings of cybersecurity and governance-related roles using unsupervised learning, then interpret the clusters as possible governance role domains such as audit, compliance, risk, security engineering, and leadership.

**Dataset:** Kaggle — *Salary Cyber Security Jobs*  
Source: `https://www.kaggle.com/datasets/dannyrevaldo/salary-cyber-security-jobs`

**Assignment coverage:**  
- Part I: Text clustering with preprocessing, TF-IDF vectorization, K-means clustering, cluster interpretation, and silhouette score testing.  
- Part II: Reinforcement learning/Q-learning parameter comparison using two bots.  

> Note: For Part II, upload the Week 9 `game.py` and `bots.py` files to the same folder as this notebook if your professor requires the exact class game environment. A self-contained Q-learning backup environment is also included so the notebook remains runnable.

## Part I — Text Clustering

### 1. Dataset Selection

This project uses a new dataset that is different from the Yelp review dataset. The dataset contains cybersecurity job/salary records. For clustering, the notebook builds a text column from job-related fields such as job title, experience level, employment type, company location, remote ratio, and other available descriptive columns.

The text clustering goal is to discover role similarity patterns and translate them into a governance role assignment framework.

In [ ]:
# If running in Google Colab, install Kaggle tools if needed.
# You may need to upload your kaggle.json API token first:
#   1. Go to Kaggle > Account > Create New API Token
#   2. Upload kaggle.json in Colab
#   3. Run this cell

import os
import glob
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    import kagglehub
except ImportError:
    !pip -q install kagglehub
    import kagglehub

print("Libraries loaded.")

In [ ]:
# Download the Kaggle dataset.
# This uses kagglehub, which is usually easier in Colab than manual Kaggle API commands.

dataset_handle = "dannyrevaldo/salary-cyber-security-jobs"

try:
    dataset_path = kagglehub.dataset_download(dataset_handle)
    print("Dataset downloaded to:", dataset_path)
except Exception as e:
    print("Automatic Kaggle download failed.")
    print("Reason:", e)
    print("
Manual fallback:")
    print("1. Download the dataset from Kaggle.")
    print("2. Upload the CSV file into this notebook environment.")
    print("3. Update dataset_path manually to the uploaded file location.")
    dataset_path = "."

csv_files = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
print("CSV files found:", csv_files)

if len(csv_files) == 0:
    raise FileNotFoundError("No CSV file found. Please upload the dataset CSV and re-run.")

In [ ]:
# Load and inspect the dataset

csv_path = csv_files[0]
df = pd.read_csv(csv_path)

print("Loaded file:", csv_path)
print("Dataset shape:", df.shape)
display(df.head())

print("
Columns:")
print(df.columns.tolist())

In [ ]:
# Identify text-friendly columns and create a clustering text column.
# Some salary datasets are mostly structured, so this cell creates a text profile from available columns.

possible_text_cols = [
    col for col in df.columns
    if df[col].dtype == "object" or str(df[col].dtype).startswith("category")
]

print("Potential text columns:", possible_text_cols)

df["role_text"] = df[possible_text_cols].fillna("").astype(str).agg(" ".join, axis=1)

extra_cols = [col for col in df.columns if col not in possible_text_cols and col != "role_text"]
for col in extra_cols:
    if col.lower() in ["remote_ratio", "work_year", "salary_in_usd", "salary"]:
        df["role_text"] = df["role_text"] + " " + col + "_" + df[col].astype(str)

print("Text column used for clustering: role_text")
display(df[["role_text"]].head())
print("Number of empty text records:", (df["role_text"].str.strip() == "").sum())

### 2. Text Preprocessing

The assignment requires tokenization, stop-word removal, punctuation/contraction removal, stemming, and examples of the output after each stage. This section uses NLTK tokenization, NLTK stop words, and Snowball stemming.

In [ ]:
# NLTK setup

import re
import string
import nltk

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

stemmer = SnowballStemmer("english")
base_stop_words = set(stopwords.words("english"))
custom_stop_words = {
    "salary", "usd", "year", "work", "job", "role", "company", "employee",
    "employment", "type", "remote", "ratio", "cyber", "security", "level",
    "full", "time", "contract", "freelance", "part", "nan", "none"
}
stop_words = base_stop_words.union(custom_stop_words)

def clean_basic(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_text(text):
    return word_tokenize(clean_basic(text))

def remove_stopwords(tokens):
    return [token for token in tokens if token not in stop_words and len(token) >= 3 and token not in string.punctuation]

def stem_tokens(tokens):
    return [stemmer.stem(token) for token in tokens]

def preprocess_pipeline(text):
    tokens = tokenize_text(text)
    no_stop = remove_stopwords(tokens)
    stems = stem_tokens(no_stop)
    return " ".join(stems)

df["tokens"] = df["role_text"].apply(tokenize_text)
df["tokens_no_stop"] = df["tokens"].apply(remove_stopwords)
df["stemmed_tokens"] = df["tokens_no_stop"].apply(stem_tokens)
df["processed_text"] = df["stemmed_tokens"].apply(lambda x: " ".join(x))

display(df[["role_text", "tokens", "tokens_no_stop", "stemmed_tokens", "processed_text"]].head(5))

In [ ]:
# Print examples after each preprocessing stage

for i in range(min(3, len(df))):
    print("=" * 100)
    print("Original text:")
    print(df.loc[i, "role_text"])
    print("
Tokenized:")
    print(df.loc[i, "tokens"])
    print("
Stop words removed:")
    print(df.loc[i, "tokens_no_stop"])
    print("
Stemmed:")
    print(df.loc[i, "stemmed_tokens"])
    print("
Concatenated processed text:")
    print(df.loc[i, "processed_text"])

### 3. Text Vectorization with TF-IDF

This section converts the cleaned role text into a TF-IDF matrix.

**Parameter choices:**
- `max_df=0.85`: removes terms that appear in more than 85% of records because overly common words do not help separate clusters.
- `min_df=2`: removes terms appearing in only one record because rare one-off terms can add noise.
- `max_features=1000`: limits the model to the most informative features and keeps the matrix manageable.
- `ngram_range=(1, 2)`: includes unigrams and bigrams so the model can capture individual words and short phrases.
- `token_pattern=r"(?u)\w\w\w+"`: keeps tokens with at least three characters, as required by the assignment.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_df=0.85,
    min_df=2,
    max_features=1000,
    ngram_range=(1, 2),
    token_pattern=r"(?u)\w\w\w+"
)

tfidf_matrix = vectorizer.fit_transform(df["processed_text"])
feature_names = vectorizer.get_feature_names_out()

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("
First 20 extracted terms:")
print(feature_names[:20])

In [ ]:
# Create a DataFrame with the transpose of the TF-IDF matrix and print it.

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=feature_names,
    index=[f"doc_{i}" for i in range(tfidf_matrix.shape[0])]
)

tfidf_transposed = tfidf_df.T

print("Transposed TF-IDF DataFrame shape:", tfidf_transposed.shape)
display(tfidf_transposed.head(20))

### 4. K-means Clustering

The assignment requires starting with an initial number of clusters, fitting K-means, retrieving cluster labels, creating a document-cluster DataFrame, displaying cluster distribution, finding top terms per cluster, and interpreting whether the clusters make sense.

Because this project is framed as a Governance Role Assigning Framework, the initial K is set to 5 so the model can explore common governance/security work domains.

In [ ]:
from sklearn.cluster import KMeans

initial_k = 5

kmeans = KMeans(n_clusters=initial_k, random_state=42, n_init=20)
df["cluster"] = kmeans.fit_predict(tfidf_matrix)

cluster_assignment_df = pd.DataFrame({
    "document_id": df.index,
    "assigned_cluster": df["cluster"],
    "role_text_preview": df["role_text"].str[:140]
})

display(cluster_assignment_df.head(20))

print("Cluster distribution:")
display(df["cluster"].value_counts().sort_index().rename("document_count").to_frame())

In [ ]:
# Determine top 15 representative terms for each cluster

def get_top_terms_per_cluster(model, terms, top_n=15):
    order_centroids = model.cluster_centers_.argsort()[:, ::-1]
    cluster_terms = {}
    for cluster_num in range(model.n_clusters):
        top_terms = [terms[ind] for ind in order_centroids[cluster_num, :top_n]]
        cluster_terms[cluster_num] = top_terms
    return cluster_terms

cluster_terms = get_top_terms_per_cluster(kmeans, feature_names, top_n=15)

for cluster_num, terms in cluster_terms.items():
    print(f"Cluster {cluster_num} top terms:")
    print(", ".join(terms))
    print()

In [ ]:
# Governance role labeling logic based on top cluster terms.
# These names are analytical interpretations, not supervised labels.

def assign_governance_label(top_terms):
    term_string = " ".join(top_terms).lower()
    if any(term in term_string for term in ["audit", "assur", "control", "compli", "govern"]):
        return "Audit, Assurance & Governance"
    elif any(term in term_string for term in ["risk", "analyst", "complianc", "policy"]):
        return "Risk & Compliance Analysis"
    elif any(term in term_string for term in ["engineer", "architect", "cloud", "network", "devsecops"]):
        return "Security Engineering & Architecture"
    elif any(term in term_string for term in ["incident", "threat", "detect", "response", "soc"]):
        return "Security Operations & Incident Response"
    elif any(term in term_string for term in ["manager", "lead", "director", "senior", "principal"]):
        return "Leadership & Program Oversight"
    else:
        return "Cybersecurity Role Generalist"

cluster_label_map = {cluster_num: assign_governance_label(terms) for cluster_num, terms in cluster_terms.items()}
df["governance_role_domain"] = df["cluster"].map(cluster_label_map)

print("Cluster interpretation map:")
for cluster_num, label in cluster_label_map.items():
    print(f"Cluster {cluster_num}: {label}")
    print("Top terms:", ", ".join(cluster_terms[cluster_num]))
    print()

display(df[["role_text", "cluster", "governance_role_domain"]].head(20))

### 5. Cluster Analysis Interpretation

Use the displayed top terms and cluster labels above to explain whether the clusters make sense.

The interpretation should be written after running the notebook because the actual top terms depend on the downloaded dataset. A sample interpretation is generated below, but you should edit it based on the final output.

In [ ]:
print("Governance Role Clustering Interpretation
")

for cluster_num in sorted(cluster_terms.keys()):
    label = cluster_label_map[cluster_num]
    terms = ", ".join(cluster_terms[cluster_num][:8])
    count = (df["cluster"] == cluster_num).sum()
    print(
        f"Cluster {cluster_num} was interpreted as '{label}' because its most representative terms include {terms}. "
        f"This cluster contains {count} records. In a governance role assignment framework, this cluster could represent "
        f"a role family or functional domain where similar responsibilities are grouped together for clearer ownership."
    )
    print()

### 6. Optimal K with Silhouette Score

The assignment requires testing K-means from `k=2` to `k=20`, plotting silhouette scores, and identifying the optimal K.

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_scores = {}
max_k = min(20, tfidf_matrix.shape[0] - 1)

for k in range(2, max_k + 1):
    model = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = model.fit_predict(tfidf_matrix)
    score = silhouette_score(tfidf_matrix, labels)
    silhouette_scores[k] = score
    print(f"k={k}, silhouette score={score:.4f}")

optimal_k = max(silhouette_scores, key=silhouette_scores.get)
print("
Optimal K based on silhouette score:", optimal_k)
print("Best silhouette score:", silhouette_scores[optimal_k])

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(list(silhouette_scores.keys()), list(silhouette_scores.values()), marker="o")
plt.title("Silhouette Score by Number of Clusters")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.xticks(list(silhouette_scores.keys()))
plt.grid(True, linestyle="--", alpha=0.4)
plt.show()

### 7. Refit K-means with Optimal K

This optional but useful step refits the model using the best K from silhouette score and updates the governance role assignment framework.

In [ ]:
best_kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=20)
df["optimal_cluster"] = best_kmeans.fit_predict(tfidf_matrix)

best_cluster_terms = get_top_terms_per_cluster(best_kmeans, feature_names, top_n=15)
best_cluster_label_map = {cluster_num: assign_governance_label(terms) for cluster_num, terms in best_cluster_terms.items()}
df["optimal_governance_role_domain"] = df["optimal_cluster"].map(best_cluster_label_map)

print("Optimal cluster distribution:")
display(df["optimal_cluster"].value_counts().sort_index().rename("document_count").to_frame())

print("
Optimal K cluster top terms and labels:")
for cluster_num, terms in best_cluster_terms.items():
    print(f"Cluster {cluster_num}: {best_cluster_label_map[cluster_num]}")
    print(", ".join(terms))
    print()

### 8. Supplemental Hierarchical Clustering Visualization

Although the assignment specifically requires K-means, the project title includes a governance role assigning hierarchical clustering framework. This section adds hierarchical agglomerative clustering as a supplemental visualization. The clustering lecture explains that hierarchical clustering can build a tree-based taxonomy/dendrogram from documents, and cutting the dendrogram at a desired level produces clusters. This section uses that idea to visualize how governance role groups merge into larger role families.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram

sample_size = min(40, df.shape[0])
sample_df = df.sample(sample_size, random_state=42).copy()
sample_matrix = vectorizer.transform(sample_df["processed_text"]).toarray()
linked = linkage(sample_matrix, method="ward")
labels = [f"{idx}: {str(text)[:35]}" for idx, text in zip(sample_df.index, sample_df["role_text"])]

plt.figure(figsize=(18, 10))
dendrogram(linked, labels=labels, leaf_rotation=75, leaf_font_size=9, color_threshold=None)
plt.title("Supplemental Governance Role Hierarchical Clustering Dendrogram")
plt.xlabel("Sampled Cybersecurity / Governance Role Records")
plt.ylabel("Operational/Textual Dissimilarity")
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
output_csv = "governance_role_clustering_results.csv"
df.to_csv(output_csv, index=False)
print("Saved clustered results to:", output_csv)

## Part II — Reinforcement Learning Agent

The assignment requires:
1. Pick one RL training parameter such as learning rate, discount factor, or exploration rate.
2. Train two bots using two different values of that parameter while holding others the same.
3. Play with the first bot and observe performance.
4. Play with the second bot and observe performance.
5. Compare the bots and explain whether the parameter changed the training outcome.

This notebook compares **exploration rate (`epsilon`)**:
- Simon: lower exploration rate  
- Olive: higher exploration rate  

Lower exploration should make the bot rely more on what it has already learned. Higher exploration should make the bot try more random actions during training, which can help it discover better paths but may also make its behavior less consistent early on.

### Option A: Use the Week 9 `game.py` and `bots.py` Files

If your class files are available, upload `game.py` and `bots.py` into the same folder as this notebook and adapt the import cell below to match the function/class names used in your Week 9 exercise.

In [ ]:
import os

print("game.py exists:", os.path.exists("game.py"))
print("bots.py exists:", os.path.exists("bots.py"))

# Example import pattern.
# Uncomment and adjust after you upload the class files.
# from game import Game
# from bots import QLearningBot

### Option B: Self-Contained Q-learning Backup Environment

The following cells create a simple grid-world reinforcement learning task. Simon and Olive both learn how to reach a goal while avoiding a penalty state. The only parameter changed is exploration rate (`epsilon`).

In [ ]:
import random
from collections import defaultdict

class SimpleGridWorld:
    def __init__(self, size=5):
        self.size = size
        self.start = (0, 0)
        self.goal = (4, 4)
        self.trap = (2, 2)
        self.state = self.start
        self.actions = ["up", "down", "left", "right"]

    def reset(self):
        self.state = self.start
        return self.state

    def step(self, action):
        row, col = self.state
        if action == "up":
            row = max(0, row - 1)
        elif action == "down":
            row = min(self.size - 1, row + 1)
        elif action == "left":
            col = max(0, col - 1)
        elif action == "right":
            col = min(self.size - 1, col + 1)
        self.state = (row, col)
        if self.state == self.goal:
            return self.state, 10, True
        elif self.state == self.trap:
            return self.state, -10, True
        else:
            return self.state, -1, False

class QLearningAgent:
    def __init__(self, name, actions, learning_rate=0.1, discount_factor=0.9, epsilon=0.1):
        self.name = name
        self.actions = actions
        self.alpha = learning_rate
        self.gamma = discount_factor
        self.epsilon = epsilon
        self.q_table = defaultdict(lambda: {action: 0.0 for action in actions})

    def choose_action(self, state):
        if random.random() < self.epsilon:
            return random.choice(self.actions)
        q_values = self.q_table[state]
        return max(q_values, key=q_values.get)

    def update(self, state, action, reward, next_state):
        current_q = self.q_table[state][action]
        max_next_q = max(self.q_table[next_state].values())
        new_q = current_q + self.alpha * (reward + self.gamma * max_next_q - current_q)
        self.q_table[state][action] = new_q

def train_agent(agent, episodes=500):
    env = SimpleGridWorld()
    rewards = []
    steps_taken = []
    for episode in range(episodes):
        state = env.reset()
        total_reward = 0
        steps = 0
        done = False
        while not done and steps < 50:
            action = agent.choose_action(state)
            next_state, reward, done = env.step(action)
            agent.update(state, action, reward, next_state)
            state = next_state
            total_reward += reward
            steps += 1
        rewards.append(total_reward)
        steps_taken.append(steps)
    return rewards, steps_taken

def play_agent(agent, rounds=5):
    env = SimpleGridWorld()
    results = []
    original_epsilon = agent.epsilon
    agent.epsilon = 0.0
    for round_num in range(rounds):
        state = env.reset()
        path = [state]
        total_reward = 0
        done = False
        steps = 0
        while not done and steps < 30:
            action = agent.choose_action(state)
            next_state, reward, done = env.step(action)
            path.append(next_state)
            state = next_state
            total_reward += reward
            steps += 1
        results.append({
            "round": round_num + 1,
            "total_reward": total_reward,
            "steps": steps,
            "ended_at": state,
            "reached_goal": state == env.goal,
            "hit_trap": state == env.trap,
            "path": path
        })
    agent.epsilon = original_epsilon
    return pd.DataFrame(results)

print("Q-learning environment and agents created.")

In [ ]:
random.seed(42)
np.random.seed(42)

simon = QLearningAgent(name="Simon", actions=["up", "down", "left", "right"], learning_rate=0.1, discount_factor=0.9, epsilon=0.05)
olive = QLearningAgent(name="Olive", actions=["up", "down", "left", "right"], learning_rate=0.1, discount_factor=0.9, epsilon=0.30)

simon_rewards, simon_steps = train_agent(simon, episodes=500)
olive_rewards, olive_steps = train_agent(olive, episodes=500)

print("Training complete.")
print("Simon average reward, last 50 episodes:", np.mean(simon_rewards[-50:]))
print("Olive average reward, last 50 episodes:", np.mean(olive_rewards[-50:]))
print("Simon average steps, last 50 episodes:", np.mean(simon_steps[-50:]))
print("Olive average steps, last 50 episodes:", np.mean(olive_steps[-50:]))

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(pd.Series(simon_rewards).rolling(25).mean(), label="Simon: epsilon=0.05")
plt.plot(pd.Series(olive_rewards).rolling(25).mean(), label="Olive: epsilon=0.30")
plt.title("Q-learning Training Performance: Simon vs. Olive")
plt.xlabel("Episode")
plt.ylabel("Rolling Average Reward")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.show()

In [ ]:
simon_play_results = play_agent(simon, rounds=5)
print("Simon gameplay results:")
display(simon_play_results)

In [ ]:
olive_play_results = play_agent(olive, rounds=5)
print("Olive gameplay results:")
display(olive_play_results)

In [ ]:
comparison = pd.DataFrame({
    "Bot": ["Simon", "Olive"],
    "Exploration Rate": [0.05, 0.30],
    "Avg Training Reward Last 50": [np.mean(simon_rewards[-50:]), np.mean(olive_rewards[-50:])],
    "Avg Training Steps Last 50": [np.mean(simon_steps[-50:]), np.mean(olive_steps[-50:])],
    "Goal Rate During Play": [simon_play_results["reached_goal"].mean(), olive_play_results["reached_goal"].mean()],
    "Avg Play Steps": [simon_play_results["steps"].mean(), olive_play_results["steps"].mean()]
})

display(comparison)